# Ch 26 · Writing Custom Kernels and Extending MJWarp

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/26-custom-kernels-and-extending.ipynb)

Companion notebook for the chapter page `chapters/26-custom-kernels-and-extending.html` ([page](../chapters/26-custom-kernels-and-extending.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

This notebook mirrors the chapter page: every example in order, then the exercises. It runs on a GPU runtime (big budgets) and on CPU (small budgets, `ON_GPU = False`). The page's outputs were recorded on CPU; numbers on CPU are never GPU performance.

## The running project: `my_env.py` (snapshot after Ch 26)
This chapter adds the RL kernels `obs_columns`, `pack_obs`, `site_reward`, `accumulate` and `count_userdata`. The cell writes the whole module so the notebook runs in an empty Colab runtime.

In [ ]:
%%writefile my_env.py
"""my_env.py - the course's running helper module, snapshot after Chapter 26.

Ch 6 added make_worlds. Ch 8 added reset_done, nonfinite_reset, snapshot and restore.
Ch 9 added describe_overflow (Example 9.7).
Ch 10 added randomize (Example 10.9): in-place, graph-friendly friction re-randomization.
Ch 14 added sensor_slices (Example 14.3): sensor name -> (adr, dim) in d.sensordata.
Ch 24 added needs_conditional_fallback and capture_step (Example 24.4): record nsub steps (plus
optional user kernels) into one CUDA graph, with the graph-conditional fallback for CUDA < 12.4.
Ch 26 adds the RL kernels (Examples 26.1-26.3), all analyzer-style (# Model: / # Data in: / # In: /
# Out: groups, _in/_out suffixes, worldid % shape[0] for "*" Model fields) and graph-safe:
obs_columns + pack_obs (observation buffer), site_reward + accumulate (reward), and
count_userdata (a per-world counter kept in d.userdata).
Ch 30 turns this file into the environments.
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

__all__ = ["make_worlds", "reset_done", "nonfinite_reset", "snapshot", "restore",
           "describe_overflow", "randomize", "sensor_slices", "needs_conditional_fallback",
           "capture_step", "obs_columns", "pack_obs", "site_reward", "accumulate",
           "count_userdata"]


# --- Ch 6 ----------------------------------------------------------------------------------------
def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None, **sizes):
    """Compile `xml` and return (mjm, m, d) with `nworld` worlds (Ch 6 signature).

    nconmax / njmax are per world; batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model
    fields their own rows (Ch 10); any other keyword (naconmax=, ...) goes to make_data.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax, **sizes)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d


# --- Ch 8: resets --------------------------------------------------------------------------------
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done):
    """Reset the worlds whose done flag is True, then recompute derived quantities.

    `done` is a wp.array of bool with shape (nworld,). Everything stays on the device, so this
    can be captured in a CUDA graph together with mjw.step (Ch 24, Ch 26).
    """
    mjw.reset_data(m, d, reset=done)
    mjw.forward(m, d)                     # reset_data does not recompute xpos, sensors, ...


def nonfinite_reset(m, d, done):
    """OR `done` with "qpos or qvel has a NaN/Inf" (MJWarp has no mj_checkPos/mj_checkVel)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- Ch 8: state snapshots -----------------------------------------------------------------------
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) float array.

    Use State.PHYSICS for MPC (it includes HISTORY and PLASTIC; note it has no TIME bit),
    add State.WARMSTART for bitwise replays with contacts, State.INTEGRATION for everything.
    """
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (only where the per-world bool mask `active` is True, if given)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


# --- Ch 9: overflow ------------------------------------------------------------------------------
def describe_overflow(d, max_worlds=8):
    """Decode d.overflow: "none", or one line per flagged world plus a world count per bit.

    Loops over mjw.OverflowType (Python >= 3.11 iterates the single-bit members only), so bits
    added by a later MJWarp release are decoded without editing this function.
    """
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    lines = []
    flagged = np.nonzero(ov)[0]
    for w in flagged[:max_worlds]:
        names = [bit.name for bit in mjw.OverflowType if ov[w] & bit]
        lines.append(f"world {w}: " + " | ".join(names))
    if len(flagged) > max_worlds:
        lines.append(f"... and {len(flagged) - max_worlds} more flagged worlds")
    counts = [f"{bit.name} x{int(np.count_nonzero(ov & bit))}" for bit in mjw.OverflowType
              if (ov & bit).any()]
    lines.append("worlds per bit: " + ", ".join(counts))
    return "\n".join(lines)


# --- Ch 10: domain randomization ----------------------------------------------------------------
@wp.kernel
def _randomize_friction_kernel(
    seed: wp.array(dtype=int),          # device array: a graph replays with the CURRENT value
    episode: wp.array(dtype=int),       # per-world episode counter, bumped on every re-draw
    reset: wp.array(dtype=bool),        # which worlds to re-draw (all if every entry is True)
    lo: float,
    hi: float,
    geom_friction: wp.array2d(dtype=wp.vec3),   # m.geom_friction, shape (nworld, ngeom)
):
    worldid, geomid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible and different every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    mu = wp.randf(rng, lo, hi)
    f = geom_friction[worldid, geomid]
    geom_friction[worldid, geomid] = wp.vec3(mu, f[1], f[2])


@wp.kernel
def _bump_episode_kernel(reset: wp.array(dtype=bool), episode: wp.array(dtype=int)):
    worldid = wp.tid()
    if reset[worldid]:
        episode[worldid] = episode[worldid] + 1


def randomize(m, seed, episode, reset, lo=0.1, hi=1.0):
    """Draw a new sliding friction in [lo, hi) for every geom of the worlds marked in reset.

    Writes m.geom_friction in place (it must already have one row per world, e.g. from
    put_model(batch_sizes={"geom_friction": nworld})), so a captured CUDA graph that contains
    this call keeps working: only array CONTENTS change, never array objects.
    seed: wp.array(dtype=int, shape=(1,)); episode: wp.array(dtype=int, shape=(nworld,));
    reset: wp.array(dtype=bool, shape=(nworld,)).
    """
    nworld = reset.shape[0]
    assert m.geom_friction.shape[0] == nworld, "batch geom_friction to nworld first"
    wp.launch(_randomize_friction_kernel, dim=(nworld, m.ngeom),
              inputs=[seed, episode, reset, lo, hi], outputs=[m.geom_friction])
    wp.launch(_bump_episode_kernel, dim=nworld, inputs=[reset], outputs=[episode])


# --- Ch 14: sensors ------------------------------------------------------------------------------
def sensor_slices(mjm):
    """Return {sensor name: (adr, dim)} so d.sensordata[:, adr:adr + dim] is that sensor.

    Unnamed sensors get the key "sensor<i>". The addresses come from the compiled MjModel
    (mjm.sensor_adr, mjm.sensor_dim) and are the same in every world.
    """
    out = {}
    for i in range(mjm.nsensor):
        name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_SENSOR, i) or f"sensor{i}"
        out[name] = (int(mjm.sensor_adr[i]), int(mjm.sensor_dim[i]))
    return out


# --- Ch 24: CUDA graphs ---------------------------------------------------------------------------
def needs_conditional_fallback(device=None):
    """True when `device` is a CUDA device whose toolkit or driver is older than CUDA 12.4.

    There, CUDA graph conditional nodes (wp.capture_while / wp.capture_if) do not exist, and
    put_model only *warns* (warp_util.check_toolkit_driver) while m.opt.graph_conditional stays True.
    On the CPU device the answer is False: Warp 1.19 records CPU graphs with APIC, which replays
    capture_while / capture_if callbacks without CUDA.
    """
    device = wp.get_device(device)
    return device.is_cuda and not wp.is_conditional_graph_supported()


def capture_step(m, d, nsub=1, post=None, verbose=True):
    """Record `nsub` calls of mjw.step(m, d), then `post(m, d)` if given, into one graph.

    Returns the graph; replay it with wp.capture_launch(graph). Capturing records the launches,
    it does not run them, so d is not advanced. The graph holds the *addresses* of every array it
    touches and the Python decisions taken while recording (integrator, solver, iteration
    counts, flags): change array contents in place (assign, fill_, copy, kernels) and recapture
    after replacing an array or changing a Python-level option (the Ch 24 knob board).
    """
    assert nsub >= 1, "nsub must be >= 1"
    device = d.qpos.device
    if needs_conditional_fallback(device) and m.opt.graph_conditional:
        m.opt.graph_conditional = False     # fixed iteration counts instead of device-side loops
        if verbose:
            print("capture_step: CUDA < 12.4, set m.opt.graph_conditional = False")
    with wp.ScopedCapture(device=device) as capture:
        for _ in range(nsub):
            mjw.step(m, d)
        if post is not None:
            post(m, d)
    return capture.graph


# --- Ch 26: observation, reward and counter kernels -----------------------------------------------
def obs_columns(mjm, sensor_names):
    """Return (nobs, cols): cols lists the d.sensordata columns of `sensor_names`, in order.

    The observation layout is [qpos (nq) | qvel (nv) | the listed sensors]. cols is a
    wp.array(dtype=int) built once on the host from sensor_slices (Ch 14), so the kernel below
    only reads device memory and can be captured in a graph.
    """
    slices = sensor_slices(mjm)
    cols = []
    for name in sensor_names:
        adr, dim = slices[name]
        cols.extend(range(adr, adr + dim))
    nobs = mjm.nq + mjm.nv + len(cols)
    return nobs, wp.array(np.array(cols, dtype=np.int32), dtype=int)


@wp.kernel
def _pack_obs_kernel(
    # Model:
    nq: int,
    nv: int,
    # Data in:
    qpos_in: wp.array2d[float],
    qvel_in: wp.array2d[float],
    sensordata_in: wp.array2d[float],
    # In:
    sensor_cols: wp.array[int],
    # Out:
    obs_out: wp.array2d[float],
):
    worldid, k = wp.tid()                 # one thread per (world, observation entry)
    if k < nq:
        obs_out[worldid, k] = qpos_in[worldid, k]
    elif k < nq + nv:
        obs_out[worldid, k] = qvel_in[worldid, k - nq]
    else:
        obs_out[worldid, k] = sensordata_in[worldid, sensor_cols[k - nq - nv]]


def pack_obs(m, d, sensor_cols, obs):
    """obs[w] = [qpos[w] | qvel[w] | sensordata[w, sensor_cols]]; obs is (nworld, nobs), preallocated."""
    assert obs.shape == (d.nworld, m.nq + m.nv + sensor_cols.shape[0]), obs.shape
    wp.launch(_pack_obs_kernel, dim=obs.shape, inputs=[m.nq, m.nv, d.qpos, d.qvel, d.sensordata,
              sensor_cols], outputs=[obs])


@wp.kernel
def _site_reward_kernel(
    # Model:
    opt_timestep: wp.array[float],
    # Data in:
    ctrl_in: wp.array2d[float],
    site_xpos_in: wp.array2d[wp.vec3],
    # In:
    siteid: int,
    target: wp.vec3,
    ctrl_weight: float,
    # Out:
    reward_out: wp.array[float],
):
    worldid = wp.tid()
    timestep = opt_timestep[worldid % opt_timestep.shape[0]]     # "*" field: maybe batched
    dist = wp.length(site_xpos_in[worldid, siteid] - target)
    ctrl_cost = float(0.0)                                      # typed: mutable in the loop
    for i in range(ctrl_in.shape[1]):
        c = ctrl_in[worldid, i]
        ctrl_cost += c * c
    reward_out[worldid] = -(dist + ctrl_weight * ctrl_cost) * timestep


def site_reward(m, d, siteid, target, ctrl_weight, reward):
    """reward[w] = -(|site_xpos[w, siteid] - target| + ctrl_weight * |ctrl[w]|^2) * timestep."""
    wp.launch(_site_reward_kernel, dim=d.nworld,
              inputs=[m.opt.timestep, d.ctrl, d.site_xpos, siteid, wp.vec3(*target), ctrl_weight],
              outputs=[reward])


@wp.kernel
def _accumulate_kernel(
    # In:
    value_in: wp.array[float],
    # Out:
    total_out: wp.array[float],
):
    worldid = wp.tid()
    total_out[worldid] += value_in[worldid]     # one thread per entry: no atomics needed


def accumulate(value, total):
    """total[w] += value[w] (episode return, running sums)."""
    wp.launch(_accumulate_kernel, dim=value.shape[0], inputs=[value], outputs=[total])


@wp.kernel
def _count_userdata_kernel(
    # In:
    col: int,
    # Data out:
    userdata_out: wp.array2d[float],
):
    worldid = wp.tid()
    userdata_out[worldid, col] += 1.0


def count_userdata(d, col=0):
    """d.userdata[:, col] += 1 in every world (needs <size nuserdata="..."/> in the MJCF).

    reset_data zeroes d.userdata, so this counts steps of the CURRENT episode; keep counters that
    must survive resets (episode numbers) in your own wp.array.
    """
    assert d.userdata.shape[1] > col, "add <size nuserdata=...> to the model"
    wp.launch(_count_userdata_kernel, dim=d.nworld, inputs=[col], outputs=[d.userdata])

## 1-2. A reward kernel and an accumulator in the step graph
The kernel follows MJWarp's conventions (`# Model:` / `# Data in:` / `# In:` / `# Out:`, `_in`/`_out`, `worldid % shape[0]`, typed locals). Everything the graph touches is allocated before capture.

In [ ]:
# Example 26.1 - ex1_reward_graph.py
"""Example 26.1 - a reward kernel and an accumulate kernel, captured with mjw.step in one graph.

The reward kernel follows MJWarp's own conventions: parameters grouped under # Model: /
# Data in: / # In: / # Out:, Data inputs end in _in, outputs in _out, the "*" Model field
opt_timestep is read with worldid % shape[0], locals are typed (ctrl_cost = float(0.0)) and no
parameter has a default. Both kernels are added to my_env.py as site_reward and accumulate.
Every world holds the double pendulum's shoulder with a different constant torque; the reward
is -(distance from the tip site to a target + 0.01 ctrl^2) * timestep, summed over the episode.
Run: python ex1_reward_graph.py [--dump-json PATH]
  --dump-json PATH writes decimated tip positions and rewards of every world (Rule D).
"""
import argparse
import json
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step26 as my_env          # examples/26-custom-kernels-and-extending/

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 200                                  # 2 s of simulated time
TARGET = (0.6, 0.0, 0.9)                     # a point the tip can reach (link lengths 0.5 + 0.5)
CTRL_W = 0.01

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
tip = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_SITE, "tip")

# allocate EVERYTHING the graph touches before capture: the graph stores array addresses
reward = wp.zeros(NWORLD, dtype=float)
total = wp.zeros(NWORLD, dtype=float)
ctrl = np.linspace(-4.0, 4.0, NWORLD, dtype=np.float32).reshape(NWORLD, 1)   # constant torque


def restart():
    d.qpos.zero_()
    d.qvel.zero_()
    d.time.zero_()
    d.ctrl.assign(ctrl)                      # assign = same array object, new contents
    total.zero_()


def post(m, d):                              # recorded after mjw.step inside the graph
    my_env.site_reward(m, d, tip, TARGET, CTRL_W, reward)
    my_env.accumulate(reward, total)


restart()
mjw.step(m, d)                               # warm-up: compile every kernel outside capture
post(m, d)
restart()
graph = my_env.capture_step(m, d, nsub=1, post=post)
print("captured: 1 x mjw.step + reward + accumulate; total before replay =", total.numpy()[:2])
for _ in range(NSTEP):
    wp.capture_launch(graph)
wp.synchronize()

tot = total.numpy()
print(f"{NSTEP} replays = {NSTEP * mjm.opt.timestep:.1f} s | ctrl per world:", ctrl[:, 0])
print("episode return per world:", tot)
best = int(np.argmax(tot))
print(f"best world: {best} (ctrl {ctrl[best, 0]:+.2f}), return {tot[best]:.4f}")

# check the LAST reward against NumPy on the host (reads happen after the loop, never inside)
p = d.site_xpos.numpy()[:, tip]
ref = -(np.linalg.norm(p - np.array(TARGET), axis=1) + CTRL_W * ctrl[:, 0] ** 2) \
    * mjm.opt.timestep
print(f"last-step reward vs NumPy: max |diff| = {np.abs(reward.numpy() - ref).max():.1e}")

if args.dump_json:                           # eager re-run with host reads: figure data only
    restart()
    frames = []
    for i in range(NSTEP):
        mjw.step(m, d)
        post(m, d)
        if i % 4 == 0:
            tp = d.site_xpos.numpy()[:8, tip]
            r = reward.numpy()[:8] / mjm.opt.timestep
            frames.append([[round(float(tp[w, 0]), 3), round(float(tp[w, 2]), 3),
                            round(float(r[w]), 3)] for w in range(min(8, NWORLD))])
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": min(8, NWORLD), "dt": 4 * float(mjm.opt.timestep),
                   "target": [TARGET[0], TARGET[2]], "pivot": [0.0, 1.5],
                   "ctrl": [round(float(c), 3) for c in ctrl[:8, 0]],
                   "returns": [round(float(t), 4) for t in tot[:8]], "frames": frames},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## An observation kernel and `d.userdata`
`obs_columns` turns sensor names into sensordata columns once on the host; `pack_obs` fills a preallocated `(nworld, nobs)` buffer on the device. `<size nuserdata="2"/>` gives every world 2 free floats, zeroed by `reset_data` and saved by `State.USERDATA`.

In [ ]:
# Example 26.2 - ex2_obs_kernel.py
"""Example 26.2 - an observation kernel and a per-world counter kept in d.userdata.

my_env.obs_columns(mjm, names) turns sensor names into sensordata columns once, on the host
(it uses my_env.sensor_slices from Ch 14); my_env.pack_obs then fills a preallocated
(nworld, nobs) buffer with [qpos | qvel | the chosen sensors], one thread per (world, entry).
We compare it with a NumPy concatenation. The model declares <size nuserdata="2"/>, so every
world has 2 free floats in d.userdata: column 0 counts steps (my_env.count_userdata), column 1
holds the highest tip so far. userdata is part of State.USERDATA and is zeroed by reset_data.
Run: python ex2_obs_kernel.py
"""
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step26 as my_env          # examples/26-custom-kernels-and-extending/

np.set_printoptions(precision=4, suppress=True)

# PENDULUM (shared model M2) plus one line: <size nuserdata="2"/>
PENDULUM_UD = """
<mujoco model="double_pendulum">
  <size nuserdata="2"/>
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50


@wp.kernel
def max_tip_height(
    # Data in:
    sensordata_in: wp.array2d[float],
    # In:
    zcol: int,
    # Data out:
    userdata_out: wp.array2d[float],
):
    worldid = wp.tid()
    userdata_out[worldid, 1] = wp.max(userdata_out[worldid, 1], sensordata_in[worldid, zcol])


mjm = mujoco.MjModel.from_xml_string(PENDULUM_UD)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
print("nuserdata:", m.nuserdata, "| d.userdata shape:", d.userdata.shape)

names = ["v1", "tip_pos"]                    # skip q1: it duplicates qpos[0]
nobs, cols = my_env.obs_columns(mjm, names)
print("sensor slices:", my_env.sensor_slices(mjm), "-> columns", cols.numpy(), "| nobs", nobs)
obs = wp.zeros((NWORLD, nobs), dtype=float)  # preallocated once, reused every step

qpos = np.zeros((NWORLD, mjm.nq), dtype=np.float32)
qpos[:, 0] = np.linspace(0.2, 1.6, NWORLD)   # a different start angle per world
d.qpos.assign(qpos)
zcol = int(mjm.sensor_adr[2]) + 2            # tip_pos z in sensordata

for _ in range(NSTEP):
    mjw.step(m, d)
    my_env.count_userdata(d, 0)              # userdata[:, 0] += 1
    wp.launch(max_tip_height, dim=d.nworld, inputs=[d.sensordata, zcol], outputs=[d.userdata])
my_env.pack_obs(m, d, cols, obs)
wp.synchronize()

ref = np.concatenate([d.qpos.numpy(), d.qvel.numpy(), d.sensordata.numpy()[:, cols.numpy()]],
                     axis=1)
o = obs.numpy()
print("obs shape:", o.shape, "| max |kernel - numpy| =", np.abs(o - ref).max())
print("obs world 0:", o[0])
print("userdata[:, 0] (steps):  ", d.userdata.numpy()[:, 0])
print("userdata[:, 1] (max tip z):", d.userdata.numpy()[:, 1])

# userdata is state: it is in State.USERDATA snapshots and is zeroed by reset_data
sig = mjw.State.USERDATA
buf = my_env.snapshot(m, d, mjm, sig)
print("State.USERDATA size:", mujoco.mj_stateSize(mjm, int(sig)), "| snapshot w0:", buf.numpy()[0])
mask = wp.array(np.arange(NWORLD) % 2 == 0, dtype=bool)
mjw.reset_data(m, d, reset=mask)             # reset the even worlds only
print("after reset_data(even worlds):", d.userdata.numpy()[:4, 0])
my_env.restore(m, d, mjm, buf, sig)
print("after restore(USERDATA):      ", d.userdata.numpy()[:4, 0])

## Step, done, reset and re-randomize in one graph
The Ch 8 auto-reset (`my_env.reset_done`) captured together with `mjw.step` by `my_env.capture_step(..., post=env_post)`.

In [ ]:
# Example 26.3 - ex3_reset_in_graph.py
"""Example 26.3 - step + done kernel + masked reset + re-randomization, all in ONE graph.

Ch 8 built the done-kernel auto-reset; Ch 24 built my_env.capture_step. Here they meet: every
replay of the graph runs mjw.step, counts the episode's steps in d.userdata[:, 0], decides
`done` on the device (pole fell, cart left the rail, or 60-step timeout), resets the done worlds
with my_env.reset_done (reset_data with a bool mask + forward) and draws a new random pole angle
for them. reset_data zeroes ctrl, qvel, act, userdata and sensordata and restores qpos to qpos0, so the
per-world push is written again after the reset by a small kernel. Nothing is read back to the host inside the loop.
Run: python ex3_reset_in_graph.py [--dump-json PATH]
  --dump-json PATH writes decimated pole angles and reset flags of 8 worlds (Rule D).
"""
import argparse
import json
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step26 as my_env          # examples/26-custom-kernels-and-extending/

np.set_printoptions(precision=4, suppress=True)

# CARTPOLE (shared model M5) plus <size nuserdata="1"/> for the per-episode step counter
CARTPOLE_UD = """
<mujoco model="cartpole">
  <size nuserdata="1"/>
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NREPLAY = 500                                # 5 s of simulated time
MAX_STEPS = 60                               # timeout: 0.6 s per episode


@wp.kernel
def done_kernel(
    # Data in:
    qpos_in: wp.array2d[float],
    userdata_in: wp.array2d[float],
    # In:
    max_angle: float,
    max_x: float,
    max_steps: float,
    # Out:
    done_out: wp.array[bool],
    reason_out: wp.array2d[int],
):
    worldid = wp.tid()
    fell = wp.abs(qpos_in[worldid, 1]) > max_angle
    off_rail = wp.abs(qpos_in[worldid, 0]) > max_x
    timeout = userdata_in[worldid, 0] >= max_steps
    done_out[worldid] = fell or off_rail or timeout
    if fell or off_rail:
        reason_out[worldid, 0] += 1          # one thread per world: plain +=, no atomics
    elif timeout:
        reason_out[worldid, 1] += 1


@wp.kernel
def randomize_pole(
    # In:
    done_in: wp.array[bool],
    seed: int,
    reason_in: wp.array2d[int],
    # Data out:
    qpos_out: wp.array2d[float],
):
    worldid = wp.tid()
    if not done_in[worldid]:
        return
    episode = reason_in[worldid, 0] + reason_in[worldid, 1]      # survives resets
    rng = wp.rand_init(seed, worldid * 100003 + episode)
    qpos_out[worldid, 1] = wp.randf(rng, -0.3, 0.3)


@wp.kernel
def apply_push(
    # In:
    push_in: wp.array[float],
    # Data out:
    ctrl_out: wp.array2d[float],
):
    worldid = wp.tid()
    ctrl_out[worldid, 0] = push_in[worldid]  # every step: reset_data zeroed ctrl of done worlds


mjm = mujoco.MjModel.from_xml_string(CARTPOLE_UD)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
done = wp.ones(NWORLD, dtype=bool)          # every buffer exists before capture
reason = wp.zeros((NWORLD, 2), dtype=int)   # episodes ended by: [fell/off rail, timeout]
push = wp.array(np.linspace(-0.3, 0.3, NWORLD, dtype=np.float32))  # a constant push per world
wp.launch(apply_push, dim=NWORLD, inputs=[push], outputs=[d.ctrl])      # (no policy yet)


def env_post(m, d):                          # everything after the physics step
    my_env.count_userdata(d, 0)
    wp.launch(done_kernel, dim=d.nworld, inputs=[d.qpos, d.userdata, 0.6, 1.5, float(MAX_STEPS)],
              outputs=[done, reason])
    my_env.reset_done(m, d, done)            # reset_data(reset=done) + forward
    wp.launch(randomize_pole, dim=d.nworld, inputs=[done, 7, reason], outputs=[d.qpos])
    wp.launch(apply_push, dim=d.nworld, inputs=[push], outputs=[d.ctrl])  # after the reset


wp.launch(randomize_pole, dim=NWORLD, inputs=[done, 7, reason], outputs=[d.qpos])  # episode 0
mjw.step(m, d)                               # warm-up compile (mjw.step and the env kernels)
env_post(m, d)
reason.zero_()
d.userdata.zero_()
graph = my_env.capture_step(m, d, nsub=1, post=env_post)
frames = []
done_since = np.zeros(min(8, NWORLD), dtype=bool)
for i in range(NREPLAY):
    wp.capture_launch(graph)
    if args.dump_json:                       # host reads for the figure only
        done_since |= done.numpy()[:8]       # remember resets between sampled frames
        if i % 5 == 4:
            q = d.qpos.numpy()[:8, 1]
            frames.append([[round(float(q[w]), 3), int(done_since[w])] for w in range(len(q))])
            done_since[:] = False
wp.synchronize()

r = reason.numpy()
print(f"{NREPLAY} graph replays = {NREPLAY * mjm.opt.timestep:.1f} s per world")
print("push per world (d.ctrl):  ", d.ctrl.numpy()[:8, 0])
print("episodes per world:      ", r.sum(axis=1)[:8])
print("  ended by fall/off rail:", r[:8, 0])
print("  ended by timeout:      ", r[:8, 1])
print("steps in current episode (userdata[:, 0]):", d.userdata.numpy()[:8, 0])
print("all finite:", bool(np.isfinite(d.qpos.numpy()).all()),
      "| overflow:", my_env.describe_overflow(d))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": min(8, NWORLD), "dt": 5 * float(mjm.opt.timestep),
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 3-4. Callbacks: passive forces
`m.callback.passive` is the last line of `passive.passive`: ADD to `d.qfrc_passive`. Quadratic drag gives the terminal speed `sqrt(m g / c)`.

In [ ]:
# Example 26.4 - ex4_passive_callback.py
"""Example 26.4 - quadratic air drag through m.callback.passive.

passive.py::passive calls m.callback.passive(m, d) as its very last line, after it has summed
springs, dampers, gravity compensation, fluid and adhesion into d.qfrc_passive. A passive
callback must therefore ADD to qfrc_passive (+=), never overwrite it. Our kernel applies
F = -c |v| v to the ball's three linear DOFs (a free joint's qvel[0:3] is the world-frame linear
velocity), with a different drag coefficient c per world. The ball falls from 30 m and reaches
the terminal speed v_t = sqrt(m g / c).
Run: python ex4_passive_callback.py [--dump-json PATH]
  --dump-json PATH writes vz(t) of 4 worlds every 25 steps for the page's speed plot (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 1250                                 # 2.5 s: long enough to reach terminal speed


@wp.kernel
def quadratic_drag(
    # Data in:
    qvel_in: wp.array2d[float],
    # In:
    drag: wp.array[float],
    # Data out:
    qfrc_passive_out: wp.array2d[float],
):
    worldid = wp.tid()
    c = drag[worldid % drag.shape[0]]
    v = wp.vec3(qvel_in[worldid, 0], qvel_in[worldid, 1], qvel_in[worldid, 2])
    f = -c * wp.length(v) * v
    for i in range(3):
        qfrc_passive_out[worldid, i] += f[i]   # ADD: springs/dampers are already in here


drag = wp.array(np.geomspace(0.25, 2.0, NWORLD).astype(np.float32), dtype=float)
calls = [0]


def passive_callback(m, d):
    calls[0] += 1
    wp.launch(quadratic_drag, dim=d.nworld, inputs=[d.qvel, drag], outputs=[d.qfrc_passive])


mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
m.callback.passive = passive_callback

qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = 30.0                            # high above the floor: no contact in 2.5 s
d.qpos.assign(qpos)
frames = []
for i in range(NSTEP):
    mjw.step(m, d)
    if args.dump_json and i % 25 == 24:      # host read for the figure only
        frames.append([round(float(v), 3) for v in d.qvel.numpy()[:4, 2]])
wp.synchronize()

vz = d.qvel.numpy()[:, 2]
c = drag.numpy()
vt = np.sqrt(1.0 * 9.81 / c)
print(f"callback calls: {calls[0]} for {NSTEP} steps (one per forward)")
print("drag c per world:     ", c[:4])
print("vz after 2.5 s [m/s]: ", vz[:4])
print("analytic -sqrt(mg/c): ", -vt[:4])
print("relative error:       ", np.abs(vz[:4] + vt[:4]) / vt[:4])
print("qfrc_passive[:, 2] = c vz^2 =", d.qfrc_passive.numpy()[:4, 2], "(balances m g = 9.81)")
print("height z [m]:         ", d.qpos.numpy()[:4, 2])

# without the callback the ball just free-falls: vz = -g t (2 s only: it would hit the floor)
m.callback.passive = None
d.qpos.assign(qpos)
d.qvel.zero_()
for _ in range(1000):
    mjw.step(m, d)
print("no callback, 2 s, vz:", d.qvel.numpy()[:4, 2], " (-g t =", round(-9.81 * 2.0, 3), ")")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": 4, "dt": 25 * float(mjm.opt.timestep),
                   "drag": [round(float(x), 4) for x in c[:4]], "frames": frames},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## Actuator callbacks run after the force kernel
For `gaintype="user"` the kernel leaves gain 0; `act_gain` must write the final force, and `forcerange` is not applied again. C MuJoCo clamps instead.

In [ ]:
# Example 26.5 - ex5_act_gain_callback.py
"""Example 26.5 - a USER-gain actuator filled by m.callback.act_gain, and what forcerange does NOT do.

forward.py::fwd_actuation first launches _actuator_force (ctrl clamp, gain, bias, forcerange) and
only then calls m.callback.act_dyn, act_gain and act_bias. For gaintype="user" the kernel leaves
gain = 0 ("GainType.USER: gain stays 0, modified by act_gain_callback"), so its force is 0. The
callback must write the FINAL force into d.actuator_force, and forcerange is not applied again.
Here force = custom_gain(q) * ctrl with custom_gain(q) = 2 + 3 sin(q)^2 and forcerange="-1 1".
C MuJoCo, for comparison, calls its mjcb_act_gain for the gain and then clamps to forcerange.
Run: python ex5_act_gain_callback.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ARM = """
<mujoco model="user_gain_arm">
  <option timestep="0.01"/>
  <worldbody>
    <body name="arm" pos="0 0 1">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <general name="u" joint="hinge" gaintype="user" biastype="none"
             forcerange="-1 1" forcelimited="true" ctrlrange="-2 2" ctrllimited="true"/>
  </actuator>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4


@wp.func
def custom_gain(q: float) -> float:
    s = wp.sin(q)
    return 2.0 + 3.0 * s * s


@wp.kernel
def user_gain_force(
    # Data in:
    qpos_in: wp.array2d[float],
    ctrl_in: wp.array2d[float],
    # Data out:
    actuator_force_out: wp.array2d[float],
):
    worldid = wp.tid()
    # actuator 0 drives qpos[0]; ctrl is read raw here (the kernel's clamp is not visible to us)
    actuator_force_out[worldid, 0] = custom_gain(qpos_in[worldid, 0]) * ctrl_in[worldid, 0]


def act_gain(m, d):
    wp.launch(user_gain_force, dim=d.nworld, inputs=[d.qpos, d.ctrl], outputs=[d.actuator_force])


mjm = mujoco.MjModel.from_xml_string(ARM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
print("gaintype USER:", int(mjm.actuator_gaintype[0]) == int(mujoco.mjtGain.mjGAIN_USER),
      "| forcerange:", mjm.actuator_forcerange[0])

q = np.linspace(0.0, 1.2, NWORLD, dtype=np.float32)
u = np.array([0.4, 0.8, -1.0, 1.5], dtype=np.float32)[np.arange(NWORLD) % 4]
d.qpos.assign(q.reshape(NWORLD, 1))
d.ctrl.assign(u.reshape(NWORLD, 1))

mjw.forward(m, d)
print("no callback:   actuator_force =", d.actuator_force.numpy()[:4, 0], "(gain stays 0)")

m.callback.act_gain = act_gain
mjw.forward(m, d)
f = d.actuator_force.numpy()[:, 0]
expected = (2.0 + 3.0 * np.sin(q) ** 2) * u
print("with callback: actuator_force =", f[:4])
print("formula g(q) * ctrl           =", expected[:4])
print("qfrc_actuator (gear 1)        =", d.qfrc_actuator.numpy()[:4, 0])
assert np.allclose(f, expected, atol=1e-5)
assert (np.abs(f) > 1.0).any(), "expected some force outside forcerange"
print("forcerange [-1, 1] reapplied after the callback?", bool((np.abs(f) <= 1.0).all()))

# C MuJoCo: mjcb_act_gain supplies the GAIN; force = gain * ctrl, then forcerange clamps it
mujoco.set_mjcb_act_gain(lambda mm, dd, i: 2.0 + 3.0 * np.sin(dd.qpos[0]) ** 2)
try:
    mjd = mujoco.MjData(mjm)
    c_force = []
    for w in range(4):
        mjd.qpos[0], mjd.ctrl[0] = q[w], u[w]
        mujoco.mj_forward(mjm, mjd)
        c_force.append(float(mjd.actuator_force[0]))
    print("C MuJoCo (gain cb + clamp)    =", np.array(c_force, dtype=np.float32))
finally:
    mujoco.set_mjcb_act_gain(None)

## 5. Sensor callbacks and `Stage`
Called three times per forward (POS, VEL, ACC); fill your user sensor once, at its stage. `Stage` is private API: `mujoco_warp._src.types.Stage`.

In [ ]:
# Example 26.6 - ex6_sensor_callback.py
"""Example 26.6 - a user sensor filled by m.callback.sensor(m, d, stage).

The model declares <sensor><user name="u" dim="2" needstage="pos" datatype="real"/></sensor>.
MJWarp calls m.callback.sensor three times per forward: at the end of sensor_pos (Stage.POS),
sensor_vel (Stage.VEL) and sensor_acc (Stage.ACC). The callback must check `stage` and fill its
sensor only once, at the stage it declared. Stage is not exported by mujoco_warp: import it from
mujoco_warp._src.types (private API). Our sensor reports (tip height, tip distance from the
pivot) of the double pendulum, written at sensor_adr of the user sensor.
Run: python ex6_sensor_callback.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import Stage   # private API: may change between releases

np.set_printoptions(precision=4, suppress=True)

# PENDULUM (shared model M2) with a 2-dim user sensor added after the three built-in ones
PENDULUM_USER = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
    <user name="u" dim="2" needstage="pos" datatype="real"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4


@wp.kernel
def tip_sensor(
    # Data in:
    site_xpos_in: wp.array2d[wp.vec3],
    # In:
    siteid: int,
    pivot: wp.vec3,
    adr: int,
    # Data out:
    sensordata_out: wp.array2d[float],
):
    worldid = wp.tid()
    p = site_xpos_in[worldid, siteid]
    sensordata_out[worldid, adr + 0] = p[2]                    # tip height
    sensordata_out[worldid, adr + 1] = wp.length(p - pivot)    # distance from the pivot


mjm = mujoco.MjModel.from_xml_string(PENDULUM_USER)
uid = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_SENSOR, "u")
ADR = int(mjm.sensor_adr[uid])
TIP = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_SITE, "tip")
print("nsensor:", mjm.nsensor, "| user sensor type USER:",
      int(mjm.sensor_type[uid]) == int(mujoco.mjtSensor.mjSENS_USER),
      "| needstage POS:", int(mjm.sensor_needstage[uid]) == int(mujoco.mjtStage.mjSTAGE_POS),
      "| adr", ADR, "dim", int(mjm.sensor_dim[uid]))
print("Stage members:", [(s.name, int(s)) for s in Stage])

seen = []


def sensor_callback(m, d, stage):
    seen.append(Stage(stage).name)
    if stage == Stage.POS:                   # declared needstage="pos": fill it here only
        wp.launch(tip_sensor, dim=d.nworld, inputs=[d.site_xpos, TIP, wp.vec3(0.0, 0.0, 1.5), ADR],
                  outputs=[d.sensordata])


m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
m.callback.sensor = sensor_callback

qpos = np.zeros((NWORLD, mjm.nq), dtype=np.float32)
qpos[:, 0] = np.linspace(0.0, np.pi / 2, NWORLD)   # shoulder angle per world
qpos[:, 1] = np.linspace(0.0, 1.0, NWORLD)         # elbow angle per world
d.qpos.assign(qpos)
mjw.forward(m, d)
print("callback stages in one forward:", seen)

sd = d.sensordata.numpy()
print("sensordata[:, ADR:ADR+2] (tip z, |tip - pivot|):")
print(sd[:, ADR:ADR + 2])
# check against the framepos sensor of the same site (sensordata columns 2..4)
tp = sd[:, 2:5]
ref = np.stack([tp[:, 2], np.linalg.norm(tp - np.array([0.0, 0.0, 1.5]), axis=1)], axis=1)
print("max |callback - framepos-derived| =", f"{np.abs(sd[:, ADR:ADR + 2] - ref).max():.1e}")

m.opt.disableflags |= mjw.DisableBit.SENSOR
seen.clear()
mjw.forward(m, d)
print("with DisableBit.SENSOR, callback stages:", seen)

## The contact filter
`nacon` cannot shrink: clear the `ContactType.CONSTRAINT` bit of `d.contact.type` to drop a contact. The kernel launches over `naconmax`.

In [ ]:
# Example 26.7 - ex7_contactfilter.py
"""Example 26.7 - drop contacts with m.callback.contactfilter by clearing the CONSTRAINT bit.

collision_driver.collision calls m.callback.contactfilter(m, d) after the narrowphase (and the
surface-velocity kernel), before make_constraint. The contact pool is append-only: nacon cannot
shrink and slots cannot be removed. To drop a contact, clear the ContactType.CONSTRAINT bit in
d.contact.type; constraint.py skips every contact with `if not (type_in[conid] &
ContactType.CONSTRAINT)`, so it gets no efc rows. Our filter launches over naconmax (the pool
capacity; threads past nacon exit) and drops every contact that touches a geom in group 3.
Scene: two boxes on a floor; the right box is in group 3, so with the filter it falls through.
Run: python ex7_contactfilter.py [--dump-json PATH]
  --dump-json PATH writes the pool (worldid, geom pair, type per slot) and nefc per world (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import ContactType   # private API: may change between releases

np.set_printoptions(precision=4, suppress=True)

TWO_BOXES = """
<mujoco model="two_boxes">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="solid" pos="-0.3 0 0.099"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="ghost" pos="0.3 0 0.099"><freejoint/>
      <geom type="box" size=".1 .1 .1" group="3" rgba="1 .3 .3 .5"/></body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 100                                  # 0.5 s


@wp.kernel
def drop_group(
    # Model:
    geom_group: wp.array[int],
    # Data in:
    contact_geom_in: wp.array[wp.vec2i],
    nacon_in: wp.array[int],
    # In:
    group: int,
    # Data out:
    contact_type_out: wp.array[int],
):
    conid = wp.tid()                          # launched over naconmax: exit past nacon
    if conid >= nacon_in[0]:
        return
    g = contact_geom_in[conid]
    if geom_group[g[0]] == group or geom_group[g[1]] == group:
        t = contact_type_out[conid]
        contact_type_out[conid] = t & (~int(ContactType.CONSTRAINT))   # keep SENSOR/PASSIVE


def contactfilter(m, d):
    wp.launch(drop_group, dim=d.naconmax, inputs=[m.geom_group, d.contact.geom, d.nacon, 3],
              outputs=[d.contact.type])


mjm = mujoco.MjModel.from_xml_string(TWO_BOXES)
m = mjw.put_model(mjm)
ghost = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "ghost")
dump = {"naconmax": 0, "runs": {}}

for use_filter in (False, True):
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
    m.callback.contactfilter = contactfilter if use_filter else None
    mjw.forward(m, d)
    n = d.nacon.numpy()[0]
    types = d.contact.type.numpy()[:n]
    label = "with filter   " if use_filter else "without filter"
    print(f"{label}: nacon {n} | nefc {d.nefc.numpy()[:4]} | "
          f"type[:16] {types[:16]} | CONSTRAINT {int((types & 1).sum())}")
    dump["naconmax"] = int(d.naconmax)
    dump["runs"]["filter" if use_filter else "nofilter"] = {
        "worldid": d.contact.worldid.numpy()[:n].tolist(), "type": types.tolist(),
        "geom": d.contact.geom.numpy()[:n].tolist(), "nefc": d.nefc.numpy()[:2].tolist()}
    for _ in range(NSTEP):
        mjw.step(m, d)
    z = d.xpos.numpy()[:, ghost, 2]
    print(f"{'':16}after {NSTEP} steps: ghost box z = {z[:4]} | overflow: "
          f"{'none' if not d.overflow.numpy().any() else d.overflow.numpy()}")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. User-populated contacts
`m.opt.run_collision_detection = False` skips collision and keeps whatever is in `d.contact` and `d.nacon`. Never write `contact.flex/elem/vert` when `nflex == 0`. Tip: `wp.config.mode = "debug"` (set before the first launch) adds bounds checks while you develop.

In [ ]:
# Example 26.8 - ex8_user_contacts.py
"""Example 26.8 - feed your own contacts to the solver with opt.run_collision_detection=False.

With run_collision_detection=False, fwd_position skips collision_driver.collision entirely and
make_constraint reads whatever is in d.contact.* and d.nacon. Our kernel writes one contact per
world (ball on the floor) into exactly the fields the constraint builder needs:
dist, pos, frame, dim, geom, worldid, friction, solref, solimp, includemargin and type = 1
(ContactType.CONSTRAINT). It never touches d.contact.flex/elem/vert: with nflex == 0 those arrays
have shape (0,), and an out-of-bounds write crashes the CPU process or corrupts GPU memory.
We compare the solver force with C MuJoCo's own collision, then with DisableBit.CONTACT.
Run: python ex8_user_contacts.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import vec5   # private API: may change between releases

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

NWORLD = 2          # one contact per world: small on purpose, same on GPU
Z0 = 0.099          # ball centre 1 mm below "touching": dist = -0.001 (a contact needs dist < 0)


@wp.kernel
def write_ball_contacts(
    # Data in:
    qpos_in: wp.array2d[float],
    # In:
    radius: float,
    floor_geom: int,
    ball_geom: int,
    friction: vec5,
    solref: wp.vec2,
    solimp: vec5,
    # Data out:
    contact_dist_out: wp.array[float],
    contact_pos_out: wp.array[wp.vec3],
    contact_frame_out: wp.array[wp.mat33],
    contact_includemargin_out: wp.array[float],
    contact_friction_out: wp.array[vec5],
    contact_solref_out: wp.array[wp.vec2],
    contact_solreffriction_out: wp.array[wp.vec2],
    contact_solimp_out: wp.array[vec5],
    contact_dim_out: wp.array[int],
    contact_geom_out: wp.array[wp.vec2i],
    contact_worldid_out: wp.array[int],
    contact_type_out: wp.array[int],
    nacon_out: wp.array[int],
):
    worldid = wp.tid()
    cid = worldid                                   # slot = world: one contact per world
    z = qpos_in[worldid, 2]                         # ball centre height (free joint z)
    contact_dist_out[cid] = z - radius              # negative = penetration
    contact_pos_out[cid] = wp.vec3(qpos_in[worldid, 0], qpos_in[worldid, 1], 0.5 * (z - radius))
    # rows of frame: normal (from geom[0]=floor to geom[1]=ball), then two tangents
    contact_frame_out[cid] = wp.mat33(0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0)
    contact_includemargin_out[cid] = 0.0
    contact_friction_out[cid] = friction
    contact_solref_out[cid] = solref
    contact_solreffriction_out[cid] = wp.vec2(0.0, 0.0)   # (0, 0): friction rows use solref
    contact_solimp_out[cid] = solimp
    contact_dim_out[cid] = 3                        # condim 3: normal + 2 friction directions
    contact_geom_out[cid] = wp.vec2i(floor_geom, ball_geom)
    contact_worldid_out[cid] = worldid
    contact_type_out[cid] = 1                       # ContactType.CONSTRAINT: solver uses it
    if worldid == 0:
        nacon_out[0] = qpos_in.shape[0]             # one contact per world in total


def fill(m, d, mjm):
    c = d.contact
    fr = mjm.geom_friction[1]
    wp.launch(write_ball_contacts, dim=d.nworld,
              inputs=[d.qpos, 0.1, 0, 1,                       # radius, floor geom, ball geom
                      vec5(fr[0], fr[0], fr[1], fr[2], fr[2]),  # MuJoCo's 5-entry contact friction
                      wp.vec2(*mjm.geom_solref[1]), vec5(*mjm.geom_solimp[1])],
              outputs=[c.dist, c.pos, c.frame, c.includemargin, c.friction, c.solref,
                       c.solreffriction, c.solimp, c.dim, c.geom, c.worldid, c.type, d.nacon])


mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
m.opt.run_collision_detection = False             # collision is now YOUR job
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
print("nflex:", m.nflex, "| contact.flex shape:", d.contact.flex.shape,
            "| contact.efc_address shape:", d.contact.efc_address.shape)

qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = Z0
d.qpos.assign(qpos)

fill(m, d, mjm)                                   # write the contacts ...
mjw.forward(m, d)                                 # ... then forward: no collision, just solve
print("user contacts:  nacon", d.nacon.numpy()[0], "| nefc per world", d.nefc.numpy())
f = d.efc.force.numpy()
print("efc.force world 0 rows 0-3:", f[0, :4], " sum", round(float(f[0, :4].sum()), 4))
print("contact.efc_address[0] (filled by make_constraint):", d.contact.efc_address.numpy()[0])

# reference: C MuJoCo with its own collision detection in the same state
mjd = mujoco.MjData(mjm)
mjd.qpos[:] = qpos[0]
mujoco.mj_forward(mjm, mjd)
print("C MuJoCo:       ncon", mjd.ncon, "| nefc", mjd.nefc, "| efc_force", mjd.efc_force[:4],
            " sum", round(float(mjd.efc_force[:4].sum()), 4))
print("max |MJWarp - C| over the 4 rows:", f"{np.abs(f[0, :4] - mjd.efc_force[:4]).max():.2e}")

# the contacts survive a step: collision never runs, so nothing overwrites them
mjw.step(m, d)
print("after mjw.step: nacon", d.nacon.numpy()[0], "| nefc", d.nefc.numpy(),
            "| ball vz", d.qvel.numpy()[:, 2])

# contrast: DisableBit.CONTACT zeroes nacon at every step, user contacts or not
m.opt.run_collision_detection = True
m.opt.disableflags |= mjw.DisableBit.CONTACT
fill(m, d, mjm)
mjw.forward(m, d)
print("DisableBit.CONTACT: nacon", d.nacon.numpy()[0], "| nefc", d.nefc.numpy())
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 7. Kernel factories and `cache_kernel`
Keys are one hash per positional argument (objects with `.size` by size) plus the factory's module and name. In a notebook, a factory you redefine under the same name returns its old kernels until you restart the runtime.

In [ ]:
# Example 26.9 - ex9_factory.py
"""Example 26.9 - kernel factories specialized with wp.static and memoized by cache_kernel.

A factory is a Python function that takes compile-time parameters and returns a freshly defined
kernel. wp.static(...) evaluates its argument while Warp compiles the kernel, so the loop over
`power` is unrolled and the `if wp.static(absolute)` branch is removed from the variant that does
not need it. mujoco_warp._src.warp_util.cache_kernel (private API) memoizes the factory: the key
is one hash per POSITIONAL argument (objects with a .size attribute are keyed by their size,
lists by hash(tuple(...))) plus the factory's module and name. Nested kernels need
module="unique" (one Warp module each); enable_backward=False skips adjoint code generation.
Run: python ex9_factory.py
"""
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw  # noqa: F401  (imported for the course's standard import order)
from mujoco_warp._src.warp_util import _KERNEL_CACHE, cache_kernel   # private API

np.set_printoptions(precision=4, suppress=True)


@cache_kernel
def make_power(power: int, absolute: bool):
    @wp.kernel(module="unique", enable_backward=False)
    def kernel(
        # In:
        x_in: wp.array[float],
        # Out:
        y_out: wp.array[float],
    ):
        i = wp.tid()
        v = x_in[i]
        if wp.static(absolute):              # compiled in only when absolute=True
            v = wp.abs(v)
        r = float(1.0)
        for _ in range(wp.static(power)):    # unrolled: `power` is a compile-time constant
            r *= v
        y_out[i] = r

    return kernel


def my_keys():
    return [k for k in _KERNEL_CACHE if k[-2:] == (__name__, "make_power")]


x = wp.array(np.array([-2.0, -1.0, 0.5, 3.0], dtype=np.float32), dtype=float)
y = wp.zeros_like(x)

k3 = make_power(3, False)
k3_again = make_power(3, False)
k3_abs = make_power(3, True)
k2 = make_power(2, False)
print("make_power(3, False) is make_power(3, False):", k3 is k3_again)
print("make_power(3, False) is make_power(3, True): ", k3 is k3_abs)
print("make_power(3, False) is make_power(2, False):", k3 is k2)

for name, k in (("x^3", k3), ("|x|^3", k3_abs), ("x^2", k2)):
    wp.launch(k, dim=x.shape[0], inputs=[x], outputs=[y])
    print(f"{name:6}:", y.numpy())

print("cache keys:", my_keys())

# key rules: hash(1) == hash(True), so these two calls share one kernel object
print("make_power(2, 1) is make_power(2, True):", make_power(2, 1) is make_power(2, True))


# arrays (anything with .size) are keyed by their SIZE only: never bake array contents
@cache_kernel
def make_offset(table):
    offset = float(table[0])                 # baked into the kernel as a constant

    @wp.kernel(module="unique", enable_backward=False)
    def kernel(
        # In:
        x_in: wp.array[float],
        # Out:
        y_out: wp.array[float],
    ):
        i = wp.tid()
        y_out[i] = x_in[i] + wp.static(offset)

    return kernel


k_ten = make_offset(np.array([10.0, 0.0, 0.0]))
k_hundred = make_offset(np.array([100.0, 0.0, 0.0]))    # same size 3 -> cache hit!
wp.launch(k_hundred, dim=x.shape[0], inputs=[x], outputs=[y])
print("make_offset([100, 0, 0]) is make_offset([10, 0, 0]):", k_hundred is k_ten,
      "-> x + 100 gives", y.numpy())
# keyword arguments are not part of the key: the wrapper takes *args only
try:
    make_power(power=3, absolute=False)
except TypeError as e:
    print("keyword call:", type(e).__name__, "-", e)
print("number of make_power entries in _KERNEL_CACHE:", len(my_keys()))

## 8. Tiles: a block of threads per world
`launch_tiled`, `tile`, `tile_load`, `tile_load_indexed`, `tile_sum`, `tile_store`. On a GPU each world gets 64 lanes; on CPU one lane.

In [ ]:
# Example 26.10 - ex10_tile_contact_force.py
"""Example 26.10 - per-world reductions with tiles: launch_tiled, tile, tile_load, tile_sum,
tile_store and tile_load_indexed.

A tiled launch wp.launch_tiled(kernel, dim=nworld, block_dim=B) gives every world one BLOCK of B
threads; wp.tid() returns (worldid, lane) and the block's threads cooperate on tiles.
 1. world_normal_force: each lane sums the normal forces of a strided subset of the shared
    contact pool (contacts of its world only), wp.tile(partial) gathers the B partials and
    wp.tile_sum reduces them; checked against a NumPy bincount of mjw.contact_force.
 2. make_row_sum(n): sums one world's live efc.force rows (i < nefc; rows past nefc keep stale
    values from earlier steps, so a whole-row wp.tile_load would add them in). The bound n = njmax
    is a compile-time constant, hence a cache_kernel factory keyed by n.
 3. make_gather_sum(k): wp.tile_load_indexed gathers qpos entries by index (the box heights).
Scene: box_stack(3) with a different mass scale per world (body_mass batched). Resting stack:
floor carries 3 boxes, the next contact 2, the top 1, so the normal forces add up to 6 m g s.
On CPU block_dim is effectively 1 (one lane covers every entry); on a GPU B lanes share the work.
Run: python ex10_tile_contact_force.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.warp_util import cache_kernel   # private API

np.set_printoptions(precision=4, suppress=True)


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 200                                  # 1 s: the stack settles
BLOCK = 64 if ON_GPU else 1                  # lanes per world (CPU runs one lane anyway)


@wp.kernel(module="unique")                  # own module: it is launched with a custom block_dim
def world_normal_force(
    # Data in:
    contact_worldid_in: wp.array[int],
    nacon_in: wp.array[int],
    # In:
    force_in: wp.array[wp.spatial_vector],
    # Out:
    total_out: wp.array[float],
):
    worldid, lane = wp.tid()
    partial = float(0.0)
    for cid in range(lane, nacon_in[0], wp.block_dim()):   # strided walk over the shared pool
        if contact_worldid_in[cid] == worldid:
            partial += force_in[cid][0]                     # normal component (contact frame)
    s = wp.tile_sum(wp.tile(partial))                       # tree reduction across the block
    wp.tile_store(total_out, s, offset=worldid)             # 1-element tile -> total_out[worldid]


@cache_kernel
def make_row_sum(n: int):
    @wp.kernel(module="unique", enable_backward=False)
    def kernel(
        # Data in:
        nefc_in: wp.array[int],
        efc_force_in: wp.array2d[float],
        # Out:
        total_out: wp.array[float],
    ):
        worldid, lane = wp.tid()
        # rows past nefc keep stale forces from earlier steps: mask, then reduce
        partial = float(0.0)
        for i in range(lane, wp.min(nefc_in[worldid], wp.static(n)), wp.block_dim()):
            partial += efc_force_in[worldid, i]
        wp.tile_store(total_out, wp.tile_sum(wp.tile(partial)), offset=worldid)   # every lane

    return kernel


@cache_kernel
def make_gather_sum(k: int):
    @wp.kernel(module="unique", enable_backward=False)
    def kernel(
        # Data in:
        qpos_in: wp.array2d[float],
        # In:
        index: wp.array[int],
        # Out:
        total_out: wp.array[float],
    ):
        worldid, lane = wp.tid()
        idx = wp.tile_load(index, shape=wp.static(k))                    # the k column indices
        vals = wp.tile_load_indexed(qpos_in[worldid], idx, shape=(wp.static(k),))   # gather
        wp.tile_store(total_out, wp.tile_sum(vals), offset=worldid)

    return kernel


mjm = mujoco.MjModel.from_xml_string(box_stack(3))
m = mjw.put_model(mjm, batch_sizes={"body_mass": NWORLD, "body_inertia": NWORLD})
scale = np.linspace(1.0, 2.5, NWORLD, dtype=np.float32)
m.body_mass.assign(mjm.body_mass[None, :] * scale[:, None])
m.body_inertia.assign(mjm.body_inertia[None, :, :] * scale[:, None, None])
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
for _ in range(NSTEP):
    mjw.step(m, d)

nacon = int(d.nacon.numpy()[0])                       # host read: diagnostics only
ids = wp.array(np.arange(nacon, dtype=np.int32), dtype=int)
force = wp.zeros(nacon, dtype=wp.spatial_vector)
mjw.contact_force(m, d, ids, False, force)            # contact frame: force[:, 0] = normal

total = wp.zeros(NWORLD, dtype=float)
wp.launch_tiled(world_normal_force, dim=NWORLD, inputs=[d.contact.worldid, d.nacon, force],
                outputs=[total], block_dim=BLOCK)
efc_sum = wp.zeros(NWORLD, dtype=float)
wp.launch_tiled(make_row_sum(d.njmax), dim=NWORLD, inputs=[d.nefc, d.efc.force], outputs=[efc_sum],
                block_dim=BLOCK)
zcols = wp.array(np.array([7 * i + 2 for i in range(3)], dtype=np.int32), dtype=int)  # box z
height_sum = wp.zeros(NWORLD, dtype=float)
wp.launch_tiled(make_gather_sum(3), dim=NWORLD, inputs=[d.qpos, zcols], outputs=[height_sum],
                block_dim=BLOCK)
wp.synchronize()

f = force.numpy()
wid = d.contact.worldid.numpy()[:nacon]
ref = np.bincount(wid, weights=f[:, 0], minlength=NWORLD)
box_mass = float(mjm.body_mass[1])
print(f"nacon {nacon} ({nacon // NWORLD} per world) | block_dim {BLOCK} | mass scale {scale[:4]}")
print("tile sum of normal forces per world: ", total.numpy()[:4])
print("NumPy bincount of contact_force[:, 0]:", ref[:4].astype(np.float32))
print("analytic 6 m g s (resting stack):     ", (6 * box_mass * 9.81 * scale[:4]))
print(f"max |tile - bincount| = {np.abs(total.numpy() - ref).max():.1e}")
nefc = d.nefc.numpy()
efc_ref = np.array([d.efc.force.numpy()[w, :nefc[w]].sum() for w in range(NWORLD)])
print("masked row sum of efc.force (< nefc): ", efc_sum.numpy()[:4], "| NumPy:", efc_ref[:4])
print("tile_load_indexed sum of box heights: ", height_sum.numpy()[:4],
      "| NumPy:", d.qpos.numpy()[:4][:, [2, 9, 16]].sum(axis=1))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 10. Reuse an MJWarp `wp.func` and test it like MJWarp
`test_data.fixture`, `nworld` 1 and 2 with world 1 perturbed, outputs pre-filled with `wp.inf`, compared with MuJoCo C at 1e-5.

In [ ]:
# Example 26.11 - ex11_reuse_math_test.py
"""Example 26.11 - reuse an MJWarp wp.func in your kernel, and test it the way MJWarp tests itself.

mujoco_warp._src.math.quat_to_mat is a @wp.func (private API: it may change between releases),
so any kernel can call it. body_tilt computes each body's rotation matrix from d.xquat and its
tilt angle (between the body z axis and world z). The test follows CONTRIBUTING.md:
mujoco_warp.test_data.fixture(xml=...) builds MjModel/MjData/Model/Data, nworld 1 and 2 with
world 1 perturbed so the worlds differ, outputs pre-filled with wp.inf, and a comparison against
MuJoCo C (mujoco.mju_quat2Mat on mj_kinematics results) at rtol = atol = 1e-5.
Run: python ex11_reuse_math_test.py
"""
import io
import sys
import unittest

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from absl.testing import parameterized
from mujoco_warp import test_data                     # shipped in the wheel
from mujoco_warp._src.math import quat_to_mat         # private API: a @wp.func

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""


@wp.kernel
def body_tilt(
    # Data in:
    xquat_in: wp.array2d[wp.quat],
    # Out:
    mat_out: wp.array2d[wp.mat33],
    tilt_out: wp.array2d[float],
):
    worldid, bodyid = wp.tid()
    rot = quat_to_mat(xquat_in[worldid, bodyid])      # MuJoCo (w, x, y, z) quaternion
    mat_out[worldid, bodyid] = rot
    tilt_out[worldid, bodyid] = wp.acos(wp.clamp(rot[2, 2], -1.0, 1.0))


def tilt(m, d):
    mat = wp.full((d.nworld, m.nbody), wp.mat33(*([wp.inf] * 9)), dtype=wp.mat33)  # pre-fill inf
    out = wp.full((d.nworld, m.nbody), wp.inf, dtype=float)
    wp.launch(body_tilt, dim=(d.nworld, m.nbody), inputs=[d.xquat], outputs=[mat, out])
    return mat.numpy(), out.numpy()


class BodyTiltTest(parameterized.TestCase):
    @parameterized.parameters(1, 2)
    def test_body_tilt(self, nworld):
        mjm, mjd, m, d = test_data.fixture(xml=PLAYGROUND, keyframe="up", qvel_noise=2.0,
                                           nworld=nworld)
        qpos = np.tile(mjd.qpos, (nworld, 1))
        if nworld == 2:                               # perturb world 1: heterogeneous worlds
            qpos[1, 3:7] = [0.8, 0.2, -0.5, 0.26]     # ball orientation (w, x, y, z)
            qpos[1, 3:7] /= np.linalg.norm(qpos[1, 3:7])
            qpos[1, 10:14] = [0.6, 0.0, 0.8, 0.0]     # box orientation: 106 deg about y
            qpos[1, 14] = -1.0                        # arm hinge
            d.qpos.assign(qpos.astype(np.float32))
            mjw.fwd_kinematics(m, d)                  # recompute xquat for both worlds

        mat, out = tilt(m, d)

        for w in range(nworld):                       # per-world MuJoCo C reference
            mjd.qpos[:] = qpos[w]
            mujoco.mj_kinematics(mjm, mjd)
            for b in range(mjm.nbody):
                ref = np.zeros(9)
                mujoco.mju_quat2Mat(ref, mjd.xquat[b])
                np.testing.assert_allclose(mat[w, b].reshape(9), ref, rtol=1e-5, atol=1e-5)
                np.testing.assert_allclose(out[w, b], np.arccos(np.clip(ref[8], -1, 1)),
                                           rtol=1e-5, atol=1e-5)
        if nworld == 2:
            self.assertFalse(np.allclose(out[0], out[1]))   # the worlds really differ
        print(f"nworld={nworld}: tilt [deg] per body (world, ball, box, arm)")
        print(np.degrees(out))


if __name__ == "__main__" or "ipykernel" in sys.modules:
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(BodyTiltTest)
    log = io.StringIO()                               # runner chatter (timings) stays quiet
    result = unittest.TextTestRunner(stream=log, verbosity=0).run(suite)
    if not result.wasSuccessful():
        print(log.getvalue())
    print("tests run:", result.testsRun, "| failures:", len(result.failures),
          "| errors:", len(result.errors))

## 12. The kernel analyzer
The analyzer is in the repository, not in the wheel. Clone it first at the tag `v4.0.1`, which matches the installed wheel (about 60 MB with `--depth 1`); see the version note on the page. Then the example writes a learner file with deliberate issues and its fixed version and runs the CLI on both.

In [ ]:
!git clone --depth 1 --branch v4.0.1 https://github.com/google-deepmind/mujoco_warp
!git -C mujoco_warp log -1 --format='analyzer from commit %h (%cs)'

In [ ]:
# Example 26.12 - ex12_kernel_analyzer.py
"""Example 26.12 - run MJWarp's kernel analyzer on your own kernels (notebook-only).

The analyzer lives in the repository, not in the pip wheel:
contrib/kernel_analyzer/kernel_analyzer/cli.py. In the notebook a cell clones the repository
first (git clone --depth 1 --branch v4.0.1 https://github.com/google-deepmind/mujoco_warp, the
tag that matches the installed wheel); this script looks for
that clone in ./mujoco_warp (or in $MJW_REPO) and prints a skip note when it is absent.
It writes two learner files, my_kernel.py (a reward kernel with deliberate convention errors) and
my_kernel_fixed.py (the analyzer-clean version), runs
  python mujoco_warp/contrib/kernel_analyzer/kernel_analyzer/cli.py FILE --types <wheel types.py>
on each, and prints the issues (the CLI writes them to stderr, one per line, and exits 1).
Run: python ex12_kernel_analyzer.py
"""
import os
import pathlib
import subprocess
import sys

import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw  # noqa: F401  (standard import order; we only need its types.py path)
import mujoco_warp._src.types as mjw_types   # private API: used for its file path only

BAD = '''import warp as wp


@wp.kernel
def reward_bad(
    qpos: wp.array2d(dtype=float),
    body_mass: wp.array2d[float],
    reward: wp.array[float],
    scale: float = 1.0,
):
    worldid = wp.tid()
    mass = body_mass[worldid, 1]
    qpos[worldid, 0] = 0.0
    reward[worldid] = -scale * mass * qpos[worldid, 2]
'''

FIXED = '''import warp as wp


@wp.kernel
def reward_good(
    # Model:
    body_mass: wp.array2d[float],
    # Data in:
    qpos_in: wp.array2d[float],
    # In:
    scale: float,
    # Out:
    reward_out: wp.array[float],
):
    worldid = wp.tid()
    mass = body_mass[worldid % body_mass.shape[0], 1]
    reward_out[worldid] = -scale * mass * qpos_in[worldid, 2]
'''

repo = pathlib.Path(os.environ.get("MJW_REPO", "mujoco_warp"))
cli = repo / "contrib" / "kernel_analyzer" / "kernel_analyzer" / "cli.py"
types_py = mjw_types.__file__                 # check against the INSTALLED wheel's types.py
print("types.py:", pathlib.Path(types_py).name, "from the installed mujoco_warp wheel")

if not cli.exists():
    print(f"skipped: {cli} not found (clone the repository first; see the notebook cell)")
else:
    pathlib.Path("my_kernel.py").write_text(BAD)
    pathlib.Path("my_kernel_fixed.py").write_text(FIXED)
    for name in ("my_kernel.py", "my_kernel_fixed.py"):
        r = subprocess.run([sys.executable, str(cli), name, "--types", types_py],
                           capture_output=True, text=True)
        issues = [ln for ln in r.stderr.splitlines() if ln.startswith(name + ":")]
        print(f"\n{name}: exit code {r.returncode}, {len(issues)} issue(s)")
        for ln in issues:
            print(" ", ln)

## Exercise 26.1: contacts per world
Count the contacts of each world with a kernel launched over `d.naconmax` and assert against `np.bincount`. Which atomic is safe here, and why?

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = np.where(np.arange(NWORLD) % 2 == 0, 0.099, 0.5)   # even worlds touch the floor
d.qpos.assign(qpos)
mjw.forward(m, d)


@wp.kernel
def count_contacts(
    # Data in:
    contact_worldid_in: wp.array[int],
    nacon_in: wp.array[int],
    # Out:
    count_out: wp.array[int],
):
    conid = wp.tid()
    ...  # your code: skip slots past nacon, then count one contact for its world


count = wp.zeros(NWORLD, dtype=int)
wp.launch(count_contacts, dim=d.naconmax, inputs=[d.contact.worldid, d.nacon], outputs=[count])

n = d.nacon.numpy()[0]
ref = np.bincount(d.contact.worldid.numpy()[:n], minlength=NWORLD)
assert np.array_equal(count.numpy(), ref), (count.numpy(), ref)
print("contacts per world:", count.numpy())

In [ ]:
#@title Exercise 26.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = np.where(np.arange(NWORLD) % 2 == 0, 0.099, 0.5)   # even worlds touch the floor
d.qpos.assign(qpos)
mjw.forward(m, d)


@wp.kernel
def count_contacts(
    # Data in:
    contact_worldid_in: wp.array[int],
    nacon_in: wp.array[int],
    # Out:
    count_out: wp.array[int],
):
    conid = wp.tid()
    if conid >= nacon_in[0]:
        return
    wp.atomic_add(count_out, contact_worldid_in[conid], 1)  # integer atomics: order-free result


count = wp.zeros(NWORLD, dtype=int)
wp.launch(count_contacts, dim=d.naconmax, inputs=[d.contact.worldid, d.nacon], outputs=[count])

n = d.nacon.numpy()[0]
ref = np.bincount(d.contact.worldid.numpy()[:n], minlength=NWORLD)
assert np.array_equal(count.numpy(), ref), (count.numpy(), ref)
print("contacts per world:", count.numpy())

## Exercise 26.2: a cart-pole observation kernel
Write `[x, cos θ, sin θ, ẋ, θ̇]` per world and assert against NumPy with `atol=1e-6`.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
q = np.zeros((NWORLD, 2), dtype=np.float32)
q[:, 1] = np.linspace(-0.5, 0.5, NWORLD)
d.qpos.assign(q)
for _ in range(20):
    mjw.step(m, d)
obs = wp.zeros((NWORLD, 5), dtype=float)


@wp.kernel
def cartpole_obs(
    # Data in:
    qpos_in: wp.array2d[float],
    qvel_in: wp.array2d[float],
    # Out:
    obs_out: wp.array2d[float],
):
    worldid = wp.tid()
    ...  # your code: [x, cos(theta), sin(theta), xdot, thetadot]


wp.launch(cartpole_obs, dim=NWORLD, inputs=[d.qpos, d.qvel], outputs=[obs])

qp, qv = d.qpos.numpy(), d.qvel.numpy()
ref = np.stack([qp[:, 0], np.cos(qp[:, 1]), np.sin(qp[:, 1]), qv[:, 0], qv[:, 1]], axis=1)
assert np.allclose(obs.numpy(), ref, atol=1e-6), np.abs(obs.numpy() - ref).max()
print("obs[0]:", obs.numpy()[0].round(4), "| max diff", np.abs(obs.numpy() - ref).max())

In [ ]:
#@title Exercise 26.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
q = np.zeros((NWORLD, 2), dtype=np.float32)
q[:, 1] = np.linspace(-0.5, 0.5, NWORLD)
d.qpos.assign(q)
for _ in range(20):
    mjw.step(m, d)
obs = wp.zeros((NWORLD, 5), dtype=float)


@wp.kernel
def cartpole_obs(
    # Data in:
    qpos_in: wp.array2d[float],
    qvel_in: wp.array2d[float],
    # Out:
    obs_out: wp.array2d[float],
):
    worldid = wp.tid()
    theta = qpos_in[worldid, 1]
    obs_out[worldid, 0] = qpos_in[worldid, 0]
    obs_out[worldid, 1] = wp.cos(theta)
    obs_out[worldid, 2] = wp.sin(theta)
    obs_out[worldid, 3] = qvel_in[worldid, 0]
    obs_out[worldid, 4] = qvel_in[worldid, 1]


wp.launch(cartpole_obs, dim=NWORLD, inputs=[d.qpos, d.qvel], outputs=[obs])

qp, qv = d.qpos.numpy(), d.qvel.numpy()
ref = np.stack([qp[:, 0], np.cos(qp[:, 1]), np.sin(qp[:, 1]), qv[:, 0], qv[:, 1]], axis=1)
assert np.allclose(obs.numpy(), ref, atol=1e-6), np.abs(obs.numpy() - ref).max()
print("obs[0]:", obs.numpy()[0].round(4), "| max diff", np.abs(obs.numpy() - ref).max())

## Stretch: a custom passive spring field
A critically damped spring toward an anchor via `m.callback.passive`. Extend it: per-world anchors in `d.userdata`, capture in a graph, and check that changing the callback after capture needs a recapture.

In [ ]:
# Stretch: a custom passive spring field (a starting point; extend it as the page suggests)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
NWORLD = 4
K, ANCHOR = 50.0, (0.0, 0.0, 0.6)


@wp.kernel
def spring_field(
    # Data in:
    qpos_in: wp.array2d[float],
    qvel_in: wp.array2d[float],
    # In:
    k: float,
    anchor: wp.vec3,
    # Data out:
    qfrc_passive_out: wp.array2d[float],
):
    worldid = wp.tid()
    p = wp.vec3(qpos_in[worldid, 0], qpos_in[worldid, 1], qpos_in[worldid, 2])
    v = wp.vec3(qvel_in[worldid, 0], qvel_in[worldid, 1], qvel_in[worldid, 2])
    f = k * (anchor - p) - 2.0 * wp.sqrt(k) * v          # critically damped for mass 1
    for i in range(3):
        qfrc_passive_out[worldid, i] += f[i]


mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
m.callback.passive = lambda m, d: wp.launch(spring_field, dim=d.nworld,
    inputs=[d.qpos, d.qvel, K, wp.vec3(*ANCHOR)], outputs=[d.qfrc_passive])
for _ in range(1000):
    mjw.step(m, d)
z = d.qpos.numpy()[:, 2]
print("rest height:", z, "| predicted", f"{ANCHOR[2] - 9.81 / K:.4f}")